In [1]:
import pandas as pd
import numpy as np

### Reading every csv of the diretory 'original_datasets'

In [2]:
df_academy = pd.read_csv('originais/academy.csv')
df_box = pd.read_csv('originais/box_office.csv')
df_genre = pd.read_csv('originais/genres.csv')
df_films = pd.read_csv('originais/pixar_films.csv')
df_rating = pd.read_csv('originais/public_response.csv')

dfs = {
    "academy.csv": df_academy,
    "box_office.csv": df_box,
    "genres.csv": df_genre,
    "pixar_films.csv": df_films,
    "public_response.csv": df_rating
}

### Removing columns that won't be used

In [3]:
df_academy.drop(columns=['award_type'], inplace=True)
df_box.drop(columns=['box_office_us_canada', 'box_office_other'], inplace=True)
df_genre.drop(columns=['category'], inplace=True)
df_films.drop(columns=['run_time', 'film_rating', 'plot'], inplace=True)
df_rating = df_rating[['film', 'imdb_score']]

### Inspecting the values from each dataframe

In [4]:
for name, df in dfs.items():
    print(name)
    print('shape', df.shape)
    print(df.info())
    print()

academy.csv
shape (89, 2)
<class 'pandas.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   film    89 non-null     str  
 1   status  89 non-null     str  
dtypes: str(2)
memory usage: 1.5 KB
None

box_office.csv
shape (28, 3)
<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   film                  28 non-null     str    
 1   budget                27 non-null     float64
 2   box_office_worldwide  28 non-null     int64  
dtypes: float64(1), int64(1), str(1)
memory usage: 804.0 bytes
None

genres.csv
shape (204, 2)
<class 'pandas.DataFrame'>
RangeIndex: 204 entries, 0 to 203
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   film    204 non-null    str  
 1   value   204 non-null    str 

### Checking null values from each dataframe

In [5]:
for name, df in dfs.items():
    print(df)

            film                    status
0      Toy Story  Award not yet introduced
1      Toy Story                 Nominated
2      Toy Story                Ineligible
3      Toy Story                 Nominated
4      Toy Story                 Nominated
..           ...                       ...
84     Lightyear                Ineligible
85     Elemental                 Nominated
86     Elemental                Ineligible
87  Inside Out 2                Ineligible
88  Inside Out 2                 Nominated

[89 rows x 2 columns]
                   film       budget  box_office_worldwide
0             Toy Story   30000000.0             394436586
1          A Bug's Life  120000000.0             363258859
2           Toy Story 2   90000000.0             511358276
3        Monsters, Inc.  115000000.0             528773250
4          Finding Nemo   94000000.0             871014978
5       The Incredibles   92000000.0             631442092
6                  Cars  120000000.0            

In [6]:
for name, df in dfs.items():
    print(name)
    print(df.isna().sum())
    print()

academy.csv
film      0
status    0
dtype: int64

box_office.csv
film                    0
budget                  1
box_office_worldwide    0
dtype: int64

genres.csv
film     0
value    0
dtype: int64

pixar_films.csv
number          0
film            0
release_date    0
dtype: int64

public_response.csv
film                      0
rotten_tomatoes_score     0
rotten_tomatoes_counts    0
metacritic_score          0
metacritic_counts         0
cinema_score              3
imdb_score                0
imdb_counts               0
dtype: int64



### Inserting budget of the film Luca because the information was null in the original dataset

In [7]:
df_box.loc[df_box['film'] == 'Luca', 'budget'] = 135000000

In [8]:
for name, df in dfs.items():
    print(name, df.duplicated().sum())
    
    if df.duplicated().sum() > 0:
        dfs[name] = df.drop_duplicates()

academy.csv 25
box_office.csv 0
genres.csv 0
pixar_films.csv 0
public_response.csv 0


### Checking dtypes

In [9]:
for name, df in dfs.items():
    print(name)
    print(df.dtypes)
    print()

academy.csv
film      str
status    str
dtype: object

box_office.csv
film                        str
budget                  float64
box_office_worldwide      int64
dtype: object

genres.csv
film     str
value    str
dtype: object

pixar_films.csv
number          int64
film              str
release_date      str
dtype: object

public_response.csv
film                          str
rotten_tomatoes_score       int64
rotten_tomatoes_counts      int64
metacritic_score            int64
metacritic_counts           int64
cinema_score                  str
imdb_score                float64
imdb_counts                 int64
dtype: object



### Formating the datatype from 'release_date' column

In [10]:
df_films['release_date'] = pd.to_datetime(df_films['release_date'], errors='coerce')

In [11]:
for df, name in dfs.items():
    print(df, name)

academy.csv             film                    status
0      Toy Story  Award not yet introduced
1      Toy Story                 Nominated
2      Toy Story                Ineligible
5      Toy Story   Won Special Achievement
6   A Bug's Life  Award not yet introduced
..           ...                       ...
84     Lightyear                Ineligible
85     Elemental                 Nominated
86     Elemental                Ineligible
87  Inside Out 2                Ineligible
88  Inside Out 2                 Nominated

[64 rows x 2 columns]
box_office.csv                    film       budget  box_office_worldwide
0             Toy Story   30000000.0             394436586
1          A Bug's Life  120000000.0             363258859
2           Toy Story 2   90000000.0             511358276
3        Monsters, Inc.  115000000.0             528773250
4          Finding Nemo   94000000.0             871014978
5       The Incredibles   92000000.0             631442092
6                  Ca

### Grouping films by franchise

In [12]:
df_agroup = pd.merge(df_films, df_rating, left_on='film', right_on='film')
df_agroup = pd.merge(df_agroup, df_box, left_on='film', right_on='film')

franquias = {'Toy Story 2': 'Toy Story',
             'Toy Story ': 'Toy Story',
             'Toy Story 3': 'Toy Story',
             'Toy Story 4': 'Toy Story',
             'Lightyear': 'Toy Story',
             'Cars': 'Cars',
             'Cars 2': 'Cars',
             'Cars 3': 'Cars',
             'The Incredibles': 'The Incredibles',
             'Incredibles 2': 'The Incredibles',
             'Finding Dory': 'Finding Nemo/Dory',
             'Finding Nemo': 'Finding Nemo/Dory',
             'Inside Out': 'Inside Out 1 & 2',
             'Inside Out 2': 'Inside Out 1 & 2',
             'Monsters, Inc.': 'Monsters, Inc. & University',
             'Monsters University': 'Monsters, Inc. & University'}

def franchise(film):
    for key, value in franquias.items():
        if key == film:
            return value
        elif key != film:
            continue

df_agroup['franchise'] = df_agroup['film'].apply(franchise)
print(df_agroup)

dfs['df.csv'] = df_agroup

    number                 film release_date  imdb_score       budget  \
0        1            Toy Story   1995-11-22         8.3   30000000.0   
1        2         A Bug's Life   1998-11-25         7.2  120000000.0   
2        3          Toy Story 2   1999-11-24         7.9   90000000.0   
3        4       Monsters, Inc.   2001-11-02         8.1  115000000.0   
4        5         Finding Nemo   2003-05-30         8.2   94000000.0   
5        6      The Incredibles   2004-11-05         8.0   92000000.0   
6        7                 Cars   2006-06-09         7.2  120000000.0   
7        8          Ratatouille   2007-06-29         8.1  150000000.0   
8        9               WALL-E   2008-06-27         8.4  180000000.0   
9       10                   Up   2009-05-29         8.3  175000000.0   
10      11          Toy Story 3   2010-06-18         8.3  200000000.0   
11      12               Cars 2   2011-06-24         6.2  200000000.0   
12      13                Brave   2012-06-22       

### Saving the csv's

In [13]:
for name, df in dfs.items():
    df.to_csv(f'limpos/{name}', index=False)